<a href="https://colab.research.google.com/github/Prajwala15/MLProject/blob/master/updatedCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content

!rm -rf textvqa

!git clone https://github.com/Prajwala15/MLProject.git textvqa

%cd textvqa

!ls

/content
Cloning into 'textvqa'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 79 (delta 32), reused 46 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 2.23 MiB | 11.94 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Filtering content: 100% (2/2), 265.46 MiB | 6.54 MiB/s, done.
/content/textvqa
configs  MLProject_with_Member3.ipynb  README.md	 scripts
data	 outputs		       requirements.txt  src


In [2]:
!pip install -r requirements.txt
!pip install easyocr pytesseract opencv-python

In [3]:
%cd /content/textvqa/data

!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_test.json

!ls -lh

/content/textvqa/data
--2026-07-08 20:54:53--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.239.50.9, 18.239.50.104, 18.239.50.18, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.50.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21634937 (21M) [text/plain]
Saving to: ‘TextVQA_0.5.1_train.json’

TextVQA_0.5.1_train 100%[===================>]  20.63M  --.-KB/s    in 0.1s    

2026-07-08 20:54:53 (208 MB/s) - ‘TextVQA_0.5.1_train.json’ saved [21634937/21634937]

--2026-07-08 20:54:53--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.239.50.9, 18.239.50.104, 18.239.50.18, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.50.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3116162 (3.0M) [text/plain]
Saving to: ‘TextVQA

In [4]:
%cd /content/textvqa/data
!unzip train_val_images_1.zip
!unzip test_images.zip

/content/textvqa/data
Archive:  train_val_images_1.zip
   creating: train_images/
  inflating: train_images/0002d070329eb0fc.jpg  
  inflating: train_images/000421a4ed497ea4.jpg  
  inflating: train_images/000adfe5b817011c.jpg  
  inflating: train_images/001f5618a7b33d88.jpg  
  inflating: train_images/0029951a71460d1c.jpg  
  inflating: train_images/003c0cac1ea19d3e.jpg  
  inflating: train_images/00466f2d1b617884.jpg  
  inflating: train_images/00689c649b17c484.jpg  
  inflating: train_images/007d8de02b273320.jpg  
  inflating: train_images/00831662d2ba731a.jpg  
  inflating: train_images/0088be963439e57d.jpg  
  inflating: train_images/008e55e8866c3a92.jpg  
  inflating: train_images/00bd04a28d45fee5.jpg  
  inflating: train_images/00d13cf28f104d4e.jpg  
  inflating: train_images/00d9db3d2c186504.jpg  
  inflating: train_images/00df83bcbdcfa980.jpg  
  inflating: train_images/00e1868250bff33c.jpg  
  inflating: train_images/00fce9f060ba3884.jpg  
  inflating: train_images/01005e15c5

In [5]:
%cd /content/textvqa

/content/textvqa


In [6]:
!find data -name "*.jpg" | head

data/test_img/a38c22a8c2a186c4.jpg
data/test_img/36d96475a9b13a97.jpg
data/test_img/87ca9df397ddbb8b.jpg
data/test_img/dc84175596b5f4c7.jpg
data/test_img/cb95f2344d44709c.jpg
data/test_img/6aebc81b1fe78247.jpg
data/test_img/00ff185256c0dc67.jpg
data/test_img/c797d8ebed0e68ca.jpg
data/test_img/b0f45fbef27728bb.jpg
data/test_img/f9796848b90791a9.jpg


# **Preprocess the training images**

In [7]:
import cv2
import numpy as np
import os

os.makedirs("data/preprocessed_train", exist_ok=True)

def preprocess(img_path):

    img = cv2.imread(img_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    denoise = cv2.fastNlMeansDenoising(gray)

    enhanced = cv2.equalizeHist(denoise)

    # Rotation correction
    coords = np.column_stack(np.where(enhanced > 0))

    if len(coords) > 0:
        angle = cv2.minAreaRect(coords)[-1]

        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle

        h, w = enhanced.shape
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)

        enhanced = cv2.warpAffine(
            enhanced,
            M,
            (w, h),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_REPLICATE
        )

    return enhanced

# **Process all 800 training images**

In [ ]:
all_train = !find data/train_images -name "*.jpg"

count = 0

for img_path in all_train:

    processed = preprocess(img_path)

    filename = os.path.basename(img_path)

    cv2.imwrite(
        f"data/preprocessed_train/{filename}",
        processed
    )

    count += 1

print("Processed training images:", count)

In [ ]:
os.makedirs("data/preprocessed_test", exist_ok=True)

all_test = !find data/test_img -name "*.jpg"

count = 0

for img_path in all_test:

    processed = preprocess(img_path)

    filename = os.path.basename(img_path)

    cv2.imwrite(
        f"data/preprocessed_test/{filename}",
        processed
    )

    count += 1

print("Processed test images:", count)

In [ ]:
!rm -f outputs/ocr_cache/train_easyocr.json

!python scripts/01_run_ocr.py \
    --split train \
    --engine easyocr

In [ ]:
!rm -f outputs/ocr_cache/train_easyocr.json

!python scripts/01_run_ocr.py \
    --split test \
    --engine easyocr

In [ ]:
!find outputs -name "*.json"

In [ ]:
!sed -n '1,120p' scripts/01_run_ocr.py

In [ ]:
!grep -n "json.dump" scripts/01_run_ocr.py

In [ ]:
!sed -n '45,75p' scripts/01_run_ocr.py

In [ ]:
import json
import numpy as np

with open("outputs/ocr_cache/train_easyocr.json") as f:
    data=json.load(f)

conf=[]

for tokens in data.values():
    for t in tokens:
        conf.append(t["conf"])

print("Average Confidence:",np.mean(conf))
print("Minimum:",np.min(conf))
print("Maximum:",np.max(conf))

In [ ]:
low=[]

for tokens in data.values():

    for t in tokens:

        if t["conf"]<0.5:

            low.append(t)

print("Low confidence OCR:",len(low))

print(low[:10])

In [ ]:
question="What is written on the sign?"

img_id=list(data.keys())[0]

ocr_text=" ".join(
    [x["text"] for x in data[img_id]]
)

fusion=f"""
Question:
{question}

OCR:
{ocr_text}
"""

print(fusion)

In [ ]:
correct=0
total=0

for gt,ocr in zip(gt_answers,ocr_texts):

    if gt.lower().strip() in ocr.lower():

        correct+=1

    total+=1

print("OCR Baseline Accuracy:",
      round(correct/total*100,2),"%")